# DD — Foundation Analysis Notebook
**Data science jobs · skill traits · personality traits — the common baseline that every problem statement needs**

This notebook runs the analysis layers that are useful *whichever* objective you finally choose. It starts from file import and ends with exported, analysis-ready tables.

| # | Layer | What it produces |
|---|---|---|
| 1 | Setup and import | Files loaded (upload prompt in Colab) |
| 2 | Structural audit | Types, missingness, duplicates, key checks, domain-rule checks |
| 3 | Treatment policy | Documented, switchable rules to build analysis-ready frames |
| 4 | Feature derivation | Seniority, role family, skill flags, composites, salary midpoints |
| 5 | Univariate | Distributions, skew, outliers, normality, category frequencies |
| 6 | Bivariate | Outcome comparisons (effect sizes, tests), correlations, salary drivers |
| 7 | Consolidation (bridge layer) | Construct-level tables that connect the four files without keys |
| 8 | Multivariate | PCA and clustering of skill and personality profiles |
| 9 | Predictive baselines | Cross-validated models, importances, sensitivity checks |
| 10 | Export | CSVs, figures, zip |

**Important modelling notes**
- None of the four files can be joined row-by-row. IDs are *not* unique even inside a file, and numeric overlaps between files are coincidental (checked in section 2). All linkage is at segment or construct level.
- `salary_hike_high_or_low` and `success_classification_high_low` are assumed to be coded **1 = high, 0 = low** (the group means in section 6 confirm the direction).
- Every result is associational, from small single-company samples (JDS, SDS) and sampled postings (job files).

## 1. Setup and file import

In [ ]:
import os, re, shutil, warnings
from itertools import combinations
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

try:
    from IPython.display import display
except ImportError:
    display = print

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid', context='notebook', palette='deep')
pd.set_option('display.max_columns', 60)
pd.set_option('display.width', 220)
pd.set_option('display.float_format', lambda v: f'{v:,.3f}')

RANDOM_STATE = 42
OUT_DIR = 'outputs'
FIG_DIR = os.path.join(OUT_DIR, 'figures')
os.makedirs(FIG_DIR, exist_ok=True)

# ---------------- switches you may change, then re-run from here ----------------
JUNIOR_MAX_YRS  = 3      # min experience <= this  -> "Junior" bucket (job files)
SENIOR_MIN_YRS  = 8      # min experience >= this  -> "Senior" bucket (Analytics file)
DEDUP_ANALYTICS = True   # drop exact duplicate postings from the Analytics core set
# --------------------------------------------------------------------------------

FILES = {
    'jds': 'JDS_Skill_Traits_clean.csv',
    'sds': 'SDS_Personality_Traits_clean.csv',
    'dsj': 'DataScience_Jobs_clean.csv',
    'an' : 'Analytics_Jobs_clean.csv',
}
JDS_SKILLS = ['big_data_skills', 'maths_stats_skills', 'coding_skills',
              'ai_and_ml_skills', 'dashboard_and_storytelling_skills']
SDS_TRAITS = ['neuroticism', 'extraversion', 'openness_to_experience',
              'agreeableness', 'conscientiousness']

pretty = lambda s: s.replace('_skills', '').replace('_and_', ' & ').replace('_', ' ')

def savefig(name):
    plt.tight_layout()
    plt.savefig(os.path.join(FIG_DIR, f'{name}.png'), dpi=150, bbox_inches='tight')
    plt.show()

In [ ]:
# ---- locate the four CSVs; in Colab, prompt for upload if they are not already there ----
try:
    from google.colab import files as colab_files
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

# Optional: to read from Google Drive instead of uploading, uncomment:
# from google.colab import drive; drive.mount('/content/drive')

SEARCH_DIRS = ['.', 'data', '/content', '/content/drive/MyDrive', '/mnt/project']

def locate(fname):
    for d in SEARCH_DIRS:
        p = os.path.join(d, fname)
        if os.path.exists(p):
            return p
    return None

missing = [f for f in FILES.values() if locate(f) is None]
if missing and IN_COLAB:
    print('Please select these files in the upload dialog:\n  ' + '\n  '.join(missing))
    colab_files.upload()          # files land in the current working directory

paths = {k: locate(v) for k, v in FILES.items()}
still_missing = [FILES[k] for k, p in paths.items() if p is None]
assert not still_missing, f'Files not found: {still_missing}. Upload them or fix SEARCH_DIRS.'

jds = pd.read_csv(paths['jds'])
sds = pd.read_csv(paths['sds'])
dsj = pd.read_csv(paths['dsj'])
an  = pd.read_csv(paths['an'])

for name, df in [('JDS (junior skills)', jds), ('SDS (senior personality)', sds),
                 ('DataScience Jobs', dsj), ('Analytics Jobs', an)]:
    print(f'{name:28s} {df.shape[0]:>7,} rows x {df.shape[1]:>2} cols')

## 2. Structural audit
Per file: types, missingness, cardinality, the pre-computed `flag_*` columns, duplicates, key behaviour, and domain-rule checks.

In [ ]:
def audit(df, name):
    print('=' * 90)
    print(f'{name}: {df.shape[0]:,} rows x {df.shape[1]} columns')
    prof = pd.DataFrame({
        'dtype': df.dtypes.astype(str),
        'non_null': df.notna().sum(),
        'missing_pct': df.isna().mean() * 100,
        'n_unique': df.nunique(),
    })
    display(prof)
    flag_cols = [c for c in df.columns if c.startswith('flag_')]
    if flag_cols:
        fl = df[flag_cols].sum().to_frame('rows_flagged')
        fl['pct_of_rows'] = fl['rows_flagged'] / len(df) * 100
        print('Pre-computed quality flags:')
        display(fl)

audit(jds, 'JDS_Skill_Traits')
audit(sds, 'SDS_Personality_Traits')
audit(dsj, 'DataScience_Jobs')
audit(an,  'Analytics_Jobs')

In [ ]:
# ---- duplicates, key behaviour, and proof that the files are not relationally linked ----
DEDUP_COLS = ['job_desig', 'job_description', 'key_skills', 'experience', 'salary', 'location_raw']
dsj_content_cols = [c for c in dsj.columns if c not in ('record_no', 'reference_no') and not c.startswith('flag_')]

dups = pd.Series({
    'JDS duplicate score rows (ignoring id)': int(jds.duplicated(subset=JDS_SKILLS + ['salary_hike_high_or_low']).sum()),
    'SDS duplicate score rows (ignoring id)': int(sds.duplicated(subset=SDS_TRAITS + ['success_classification_high_low']).sum()),
    'DSJ duplicate rows (ignoring reference_no)': int(dsj.duplicated(subset=dsj_content_cols).sum()),
    'Analytics duplicate postings': int(an.duplicated(subset=DEDUP_COLS).sum()),
    'Analytics duplicates if salary band is ignored (info only)': int(an.duplicated(subset=[c for c in DEDUP_COLS if c != 'salary']).sum()),
}).to_frame('rows')
display(dups)

keys = pd.DataFrame({
    'rows':       [len(jds), len(sds), len(dsj), len(an)],
    'distinct':   [jds.id.nunique(), sds.id.nunique(), dsj.reference_no.nunique(), an.s_no.nunique()],
    'min':        [jds.id.min(), sds.id.min(), dsj.reference_no.min(), an.s_no.min()],
    'max':        [jds.id.max(), sds.id.max(), dsj.reference_no.max(), an.s_no.max()],
}, index=['JDS.id', 'SDS.id', 'DSJ.reference_no', 'Analytics.s_no'])
keys['is_unique_key'] = keys.rows == keys.distinct
print('\nCandidate key columns:')
display(keys)

id_sets = {'JDS.id': set(jds.id), 'SDS.id': set(sds.id), 'DSJ.reference_no': set(dsj.reference_no)}
overlap = [{'pair': f'{a} x {b}', 'shared_values': len(id_sets[a] & id_sets[b])} for a, b in combinations(id_sets, 2)]
print('\nNumeric overlap between ID columns across files (coincidental, NOT a join):')
display(pd.DataFrame(overlap))
print('Note: many Analytics postings are identical except for the salary band; they are kept (treated as distinct) because only fully identical rows are dropped.')
print('Conclusion: IDs repeat within files and mean different things across files -> use record_no / s_no as row labels only; link at construct level.')

In [ ]:
# ---- domain-rule checks ----
checks = {
    'JDS: skill scores outside 1-5':                int(((jds[JDS_SKILLS] < 1) | (jds[JDS_SKILLS] > 5)).any(axis=1).sum()),
    'SDS: trait value range (min .. max)':          f'{sds[SDS_TRAITS].min().min()} .. {sds[SDS_TRAITS].max().max()}',
    'JDS/SDS: outcome not in {0,1}':                int((~jds.salary_hike_high_or_low.isin([0, 1])).sum() + (~sds.success_classification_high_low.isin([0, 1])).sum()),
    'DSJ: avg outside [min, max] salary':           int(((dsj.avg_salary_lpa < dsj.min_salary_lpa) | (dsj.avg_salary_lpa > dsj.max_salary_lpa)).sum()),
    'DSJ: negative experience':                     int((dsj.min_experience < 0).sum()),
    'Analytics: exp_min > exp_max':                 int((an.exp_min_yrs > an.exp_max_yrs).sum()),
    'Analytics: salary_min > salary_max':           int((an.salary_min_lpa > an.salary_max_lpa).sum()),
    'Analytics: job_description missing (%)':       round(an.job_description.isna().mean() * 100, 1),
    'Analytics: key_skills flagged truncated (%)':  round(an.flag_skills_truncated.mean() * 100, 1),
    'Analytics: gig / data-entry postings':         int(an.flag_gig_or_data_entry_posting.sum()),
}
display(pd.Series(checks).to_frame('result'))

## 3. Treatment policy — building the analysis-ready ("core") frames
Rules are deliberately conservative and documented, because the brief rewards visible data-issue handling.

| File | Treatment | Reason |
|---|---|---|
| JDS, SDS | Keep all rows; sensitivity checks later (section 9) | Repeated IDs are *different* rows, so they are not dropped; the samples are too small to lose rows without evidence |
| DataScience Jobs | Drop placeholder companies (`Confidential`, `N+A`, `Data Entry`) | Not real employers; distorts company-level views |
| DataScience Jobs | Keep large-volume outliers (TCS, IBM...) but analyse `num_of_jobs` on a log scale | They are genuine mega-recruiters, not errors |
| Analytics Jobs | Drop gig / data-entry postings | Off-topic for data careers |
| Analytics Jobs | Drop exact duplicate postings (switch `DEDUP_ANALYTICS`) | Avoids double-counting skills demand |

In [ ]:
jds_core = jds.copy()
sds_core = sds.copy()
dsj_core = dsj[dsj.flag_placeholder_company == 0].copy()

an = an.copy()
an['flag_duplicate_posting'] = an.duplicated(subset=DEDUP_COLS, keep='first').astype(int)
an_mask = an.flag_gig_or_data_entry_posting == 0
if DEDUP_ANALYTICS:
    an_mask &= an.flag_duplicate_posting == 0
an_core = an[an_mask].copy()

treat = pd.DataFrame({
    'rows_in':   [len(jds), len(sds), len(dsj), len(an)],
    'rows_core': [len(jds_core), len(sds_core), len(dsj_core), len(an_core)],
}, index=['JDS', 'SDS', 'DataScience Jobs', 'Analytics Jobs'])
treat['rows_removed'] = treat.rows_in - treat.rows_core
treat['pct_removed'] = treat.rows_removed / treat.rows_in * 100
display(treat)

## 4. Feature derivation
The derived fields below create the shared vocabulary (seniority, role family, five skill dimensions) used to link the files later.

Skill dimensions in the Analytics file come from keyword dictionaries applied to title + key skills + description. They are **proxies**, and because most `key_skills` strings are truncated, prevalence figures are lower bounds.

In [ ]:
# ---------------- JDS ----------------
jds_core['outcome']      = jds_core.salary_hike_high_or_low.map({1: 'High hike', 0: 'Low hike'})
jds_core['tech_mean']    = jds_core[JDS_SKILLS].mean(axis=1)
jds_core['skill_spread'] = jds_core[JDS_SKILLS].std(axis=1)          # low = balanced profile
jds_core['n_skills_ge4'] = (jds_core[JDS_SKILLS] >= 4).sum(axis=1)

# ---------------- SDS ----------------
sds_core['outcome'] = sds_core.success_classification_high_low.map({1: 'High success', 0: 'Low success'})
for c in SDS_TRAITS:
    sds_core[f'z_{c}'] = (sds_core[c] - sds_core[c].mean()) / sds_core[c].std()
good_traits = ['extraversion', 'openness_to_experience', 'agreeableness', 'conscientiousness']
sds_core['n_traits_above_median'] = sum((sds_core[c] > sds_core[c].median()).astype(int) for c in good_traits)

# ---------------- DataScience Jobs ----------------
dsj_core['role_family']     = dsj_core.job_title.str.replace(r'^Senior ', '', regex=True)
dsj_core['is_senior_title'] = (dsj_core.job_title.str.startswith('Senior') | (dsj_core.job_title == 'Data Architect')).astype(int)
dsj_core['seniority']       = np.where(dsj_core.is_senior_title == 1, 'Senior', 'Junior/Mid')
dsj_core['salary_spread']   = dsj_core.max_salary_lpa - dsj_core.min_salary_lpa
dsj_core['log_jobs']        = np.log1p(dsj_core.num_of_jobs)
company_openings            = dsj_core.groupby('company_name').num_of_jobs.transform('sum')
dsj_core['large_recruiter'] = (company_openings >= company_openings.quantile(0.90)).astype(int)

# ---------------- Analytics Jobs ----------------
an_core['salary_mid'] = (an_core.salary_min_lpa + an_core.salary_max_lpa) / 2
an_core['exp_mid']    = (an_core.exp_min_yrs + an_core.exp_max_yrs) / 2
an_core['seniority']  = np.select([an_core.exp_min_yrs <= JUNIOR_MAX_YRS, an_core.exp_min_yrs >= SENIOR_MIN_YRS],
                                  ['Junior', 'Senior'], default='Mid')

def role_family(title):
    t = str(title).lower()
    if 'data scien' in t:                                              return 'Data Scientist'
    if re.search(r'machine learning|\bml\b|\bai\b|deep learning|nlp', t): return 'ML/AI Engineer'
    if re.search(r'data engineer|\betl\b|big data|hadoop|data warehouse|\bdwh\b', t): return 'Data Engineer'
    if 'architect' in t:                                               return 'Architect'
    if re.search(r'analyst|analytics', t):                             return 'Analyst'
    if re.search(r'consultant|manager|lead|head|director', t):         return 'Consultant/Manager'
    if re.search(r'developer|software|engineer|programmer', t):        return 'Software/IT'
    return 'Other'

an_core['role_family']      = an_core.job_desig.map(role_family)
CORE_ROLES                  = ['Data Scientist', 'ML/AI Engineer', 'Data Engineer', 'Architect', 'Analyst']
an_core['is_core_data_role'] = an_core.role_family.isin(CORE_ROLES).astype(int)

SKILL_PATTERNS = {
    'big_data_skills':    r'hadoop|spark|big data|bigdata|\bhive\b|kafka|hbase|nosql|mapreduce|data lake|databricks|cloudera',
    'maths_stats_skills': r'statistic|regression|forecast|probabilit|mathemat|hypothesis|quantitative|time series|bayes|optimi[sz]ation|\bsas\b|\bspss\b',
    'coding_skills':      r'python|\bjava\b|scala|\bsql\b|programming|c\+\+|pyspark|\bperl\b|javascript|coding',
    'ai_and_ml_skills':   r'machine learning|\bml\b|deep learning|artificial intelligence|\bai\b|\bnlp\b|natural language|neural|tensorflow|pytorch|computer vision|data mining|predictive model',
    'dashboard_and_storytelling_skills': r'tableau|power ?bi|dashboard|visuali[sz]|qlik|storytelling|data story|reporting|cognos|looker',
}
text_all = (an_core.job_desig.fillna('') + ' ' + an_core.key_skills.fillna('') + ' ' + an_core.job_description.fillna('')).str.lower()
for k, pat in SKILL_PATTERNS.items():
    an_core[f'skill_{k}'] = text_all.str.contains(pat, regex=True).astype(int)
an_core['n_skill_dims'] = an_core[[f'skill_{k}' for k in SKILL_PATTERNS]].sum(axis=1)

METROS = ['Bengaluru', 'Mumbai', 'Delhi NCR', 'Gurgaon', 'Noida', 'Pune', 'Hyderabad', 'Chennai', 'Kolkata']
an_core['is_metro']       = an_core.primary_location.isin(METROS).astype(int)
top_locs                  = an_core.primary_location.value_counts().head(8).index
an_core['location_group'] = np.where(an_core.primary_location.isin(top_locs), an_core.primary_location, 'Other')
an_core['desc_missing']   = an_core.job_description.isna().astype(int)

print('Role family (Analytics core):')
display(an_core.role_family.value_counts().to_frame('postings'))
print('Skill-dimension prevalence (share of postings):')
display(an_core[[f'skill_{k}' for k in SKILL_PATTERNS]].mean().rename(lambda s: pretty(s.replace('skill_', ''))).to_frame('share'))

## 5. Univariate analysis

In [ ]:
def numeric_summary(df, cols):
    out = df[cols].agg(['count', 'mean', 'std', 'min', 'median', 'max']).T
    out['skew'] = df[cols].skew()
    out['kurtosis'] = df[cols].kurt()
    q1, q3 = df[cols].quantile(.25), df[cols].quantile(.75)
    iqr = q3 - q1
    out['iqr_outliers'] = ((df[cols] < q1 - 1.5 * iqr) | (df[cols] > q3 + 1.5 * iqr)).sum()
    return out

def hist_grid(df, cols, title, ncols=3, bins=20, name=None):
    nrows = int(np.ceil(len(cols) / ncols))
    fig, axes = plt.subplots(nrows, ncols, figsize=(4.6 * ncols, 3.4 * nrows))
    axes = np.atleast_1d(axes).ravel()
    for ax, c in zip(axes, cols):
        sns.histplot(df[c].dropna(), bins=bins, kde=True, ax=ax)
        ax.set_title(pretty(c))
        ax.set_xlabel('')
    for ax in axes[len(cols):]:
        ax.axis('off')
    fig.suptitle(title, y=1.02, fontsize=14)
    savefig(name or title.lower().replace(' ', '_'))

def freq(df, col, n=10):
    vc = df[col].value_counts(dropna=False).head(n).to_frame('count')
    vc['pct'] = vc['count'] / len(df) * 100
    return vc

print('JDS skill scores'); display(numeric_summary(jds_core, JDS_SKILLS))
hist_grid(jds_core, JDS_SKILLS + ['tech_mean'], 'JDS skill distributions')
print('SDS personality traits'); display(numeric_summary(sds_core, SDS_TRAITS))
hist_grid(sds_core, SDS_TRAITS, 'SDS trait distributions')

In [ ]:
# ---- normality: decides parametric vs non-parametric tests for the small trait files ----
norm_rows = []
for df, cols, tag in [(jds_core, JDS_SKILLS, 'JDS'), (sds_core, SDS_TRAITS, 'SDS')]:
    for c in cols:
        w, p = stats.shapiro(df[c])
        norm_rows.append({'file': tag, 'variable': c, 'shapiro_W': w, 'p_value': p, 'normal_at_5pct': p > 0.05})
display(pd.DataFrame(norm_rows))
print('Skills are capped at 5 and left-skewed, so rank-based tests (Mann-Whitney, Spearman) are used from here on.')

In [ ]:
dsj_num = ['min_experience', 'avg_salary_lpa', 'min_salary_lpa', 'max_salary_lpa', 'num_of_jobs', 'salary_spread']
print('DataScience Jobs (core)'); display(numeric_summary(dsj_core, dsj_num))
hist_grid(dsj_core.assign(log_num_of_jobs=dsj_core.log_jobs), ['min_experience', 'avg_salary_lpa', 'salary_spread', 'log_num_of_jobs'],
          'DataScience Jobs distributions', ncols=2)

an_num = ['exp_min_yrs', 'exp_max_yrs', 'salary_min_lpa', 'salary_max_lpa', 'salary_mid', 'num_locations']
print('Analytics Jobs (core)'); display(numeric_summary(an_core, an_num))
hist_grid(an_core, ['exp_min_yrs', 'salary_mid', 'num_locations'], 'Analytics Jobs distributions', bins=15)

In [ ]:
# ---- categorical frequencies ----
print('DataScience Jobs: titles');  display(freq(dsj_core, 'job_title'))
print('DataScience Jobs: top companies by number of rows'); display(freq(dsj_core, 'company_name', 8))
print('Analytics: seniority');      display(freq(an_core, 'seniority'))
print('Analytics: salary band');    display(freq(an_core, 'salary'))
print('Analytics: primary location (top 10)'); display(freq(an_core, 'primary_location', 10))

fig, ax = plt.subplots(2, 2, figsize=(13, 8))
dsj_core.groupby('job_title').num_of_jobs.sum().sort_values().plot.barh(ax=ax[0, 0], title='DSJ: total openings by title')
an_core.role_family.value_counts().sort_values().plot.barh(ax=ax[0, 1], title='Analytics: postings by role family')
an_core.primary_location.value_counts().head(10).sort_values().plot.barh(ax=ax[1, 0], title='Analytics: top 10 locations')
band_order = ['0to3', '3to6', '6to10', '10to15', '15to25', '25to50']
an_core.salary.value_counts().reindex(band_order).plot.bar(ax=ax[1, 1], title='Analytics: salary band (LPA)', rot=0)
savefig('categorical_overview')

## 6. Bivariate analysis
**6a / 6b** compare high vs low outcome groups on skills (JDS) and personality (SDS) with tests and effect sizes.
**6c / 6d** explain salary in the two job files.

In [ ]:
def group_compare(df, cols, target):
    """High (1) vs Low (0): means, Mann-Whitney p (Bonferroni-adjusted), Cohen's d, AUC-style effect."""
    rows = []
    for c in cols:
        g1, g0 = df.loc[df[target] == 1, c], df.loc[df[target] == 0, c]
        u, p = stats.mannwhitneyu(g1, g0, alternative='two-sided')
        pooled = np.sqrt((g1.var(ddof=1) + g0.var(ddof=1)) / 2)
        rows.append({'feature': c, 'mean_high': g1.mean(), 'mean_low': g0.mean(),
                     'diff': g1.mean() - g0.mean(),
                     'cohens_d': (g1.mean() - g0.mean()) / pooled,
                     'auc_single': u / (len(g1) * len(g0)),
                     'p_value': p})
    out = pd.DataFrame(rows).set_index('feature')
    out['p_bonferroni'] = (out.p_value * len(cols)).clip(upper=1)
    out['significant_5pct'] = out.p_bonferroni < 0.05
    return out.sort_values('cohens_d', ascending=False)

def outcome_plots(df, cols, target_label, target_num, tag):
    long = df.melt(id_vars=target_label, value_vars=cols, var_name='feature', value_name='score')
    long['feature'] = long.feature.map(pretty)
    fig, ax = plt.subplots(1, 2, figsize=(15, 4.8), gridspec_kw={'width_ratios': [1.6, 1]})
    sns.boxplot(data=long, x='feature', y='score', hue=target_label, ax=ax[0])
    ax[0].set_title(f'{tag}: feature distributions by outcome'); ax[0].set_xlabel('')
    ax[0].tick_params(axis='x', rotation=20)
    eff = group_compare(df, cols, target_num)
    eff['label'] = eff.index.map(pretty)
    sns.barplot(data=eff, y='label', x='cohens_d', ax=ax[1], color='steelblue')
    ax[1].set_title("Effect size (Cohen's d, high vs low)"); ax[1].set_ylabel('')
    savefig(f'{tag.lower()}_outcome_comparison')

def corr_heatmap(df, cols, target, tag):
    c = df[cols + [target]].corr(method='spearman')
    mask = np.triu(np.ones_like(c, dtype=bool), k=1)
    plt.figure(figsize=(7.5, 6))
    sns.heatmap(c.rename(index=pretty, columns=pretty), mask=mask, annot=True, fmt='.2f', cmap='coolwarm', center=0, vmin=-1, vmax=1)
    plt.title(f'{tag}: Spearman correlations (incl. outcome)')
    savefig(f'{tag.lower()}_correlation')

# ---- 6a. JDS ----
jds_effects = group_compare(jds_core, JDS_SKILLS, 'salary_hike_high_or_low')
print('JDS: skills of high vs low salary-hike juniors'); display(jds_effects)
outcome_plots(jds_core, JDS_SKILLS, 'outcome', 'salary_hike_high_or_low', 'JDS')
corr_heatmap(jds_core, JDS_SKILLS, 'salary_hike_high_or_low', 'JDS')

In [ ]:
# ---- 6b. SDS ----
sds_effects = group_compare(sds_core, SDS_TRAITS, 'success_classification_high_low')
print('SDS: traits of high vs low success seniors'); display(sds_effects)
outcome_plots(sds_core, SDS_TRAITS, 'outcome', 'success_classification_high_low', 'SDS')
corr_heatmap(sds_core, SDS_TRAITS, 'success_classification_high_low', 'SDS')
print('Direction check: if the 1-group has the higher means, the 1 = high assumption holds.')

In [ ]:
# ---- 6c. DataScience Jobs: what drives salary and demand ----
order = dsj_core.groupby('job_title').avg_salary_lpa.median().sort_values().index
plt.figure(figsize=(11, 5))
sns.boxplot(data=dsj_core, y='job_title', x='avg_salary_lpa', order=order, color='lightsteelblue')
plt.title('DSJ: average salary (LPA) by job title'); plt.ylabel('')
savefig('dsj_salary_by_title')

groups = [g.avg_salary_lpa.values for _, g in dsj_core.groupby('job_title')]
H, p = stats.kruskal(*groups)
rho, p_rho = stats.spearmanr(dsj_core.min_experience, dsj_core.avg_salary_lpa)
print(f'Kruskal-Wallis salary across titles: H={H:.1f}, p={p:.2e}')
print(f'Spearman min_experience vs avg salary: rho={rho:.2f}, p={p_rho:.2e}')

prem = dsj_core.pivot_table(index='role_family', columns='seniority', values='avg_salary_lpa', aggfunc='median')
if {'Junior/Mid', 'Senior'} <= set(prem.columns):
    prem['senior_premium_pct'] = (prem['Senior'] / prem['Junior/Mid'] - 1) * 100
print('\nMedian average salary (LPA) and senior premium by role family:')
dsj_premium = prem.copy(); display(dsj_premium)

openings = dsj_core.groupby('company_name').num_of_jobs.sum().sort_values(ascending=False)
print(f'\nTop 10 companies hold {openings.head(10).sum() / openings.sum():.1%} of all openings')
display(openings.head(10).to_frame('total_openings'))

fig, ax = plt.subplots(1, 2, figsize=(12, 4.5))
sns.scatterplot(data=dsj_core, x='min_experience', y='avg_salary_lpa', hue='seniority', alpha=.5, ax=ax[0])
ax[0].set_title('DSJ: experience vs salary')
sns.scatterplot(data=dsj_core, x='log_jobs', y='avg_salary_lpa', alpha=.4, ax=ax[1], color='gray')
ax[1].set_title('DSJ: openings (log) vs salary')
savefig('dsj_experience_salary')

In [ ]:
# ---- 6d. Analytics Jobs: salary, experience, location, skills ----
for a_, b_ in [('exp_min_yrs', 'salary_min_lpa'), ('exp_min_yrs', 'salary_mid')]:
    r_, p_ = stats.spearmanr(an_core[a_], an_core[b_])
    print(f'Spearman {a_} vs {b_}: rho={r_:.2f}, p={p_:.2e}')

ct = pd.crosstab(an_core.seniority, an_core.salary, normalize='index').reindex(index=['Junior', 'Mid', 'Senior'], columns=band_order) * 100
plt.figure(figsize=(8, 3.5))
sns.heatmap(ct, annot=True, fmt='.0f', cmap='Blues'); plt.title('Analytics: % of postings in each salary band, by seniority')
savefig('an_seniority_salary_band')

role_sal = an_core[an_core.is_core_data_role == 1].groupby('role_family').salary_mid.agg(['count', 'median', 'mean']).sort_values('median', ascending=False)
print('Salary midpoint (LPA) by role family (core data roles):'); display(role_sal)
loc_sal = an_core.groupby('location_group').salary_mid.agg(['count', 'median']).sort_values('median', ascending=False)
print('Salary midpoint (LPA) by location group:'); display(loc_sal)

rows = []
for k in SKILL_PATTERNS:
    col = f'skill_{k}'
    w, wo = an_core.loc[an_core[col] == 1, 'salary_mid'], an_core.loc[an_core[col] == 0, 'salary_mid']
    row = {'skill': k, 'share_postings': an_core[col].mean(), 'median_with': w.median(), 'median_without': wo.median(),
           'diff_overall': w.median() - wo.median(), 'mannwhitney_p': stats.mannwhitneyu(w, wo).pvalue}
    for s_ in ['Junior', 'Mid', 'Senior']:
        sub = an_core[an_core.seniority == s_]
        row[f'diff_{s_}'] = sub.loc[sub[col] == 1, 'salary_mid'].median() - sub.loc[sub[col] == 0, 'salary_mid'].median()
    rows.append(row)
skill_premium = pd.DataFrame(rows).set_index('skill')
print('Salary-midpoint premium for postings mentioning each skill dimension (LPA, controlled by seniority bucket):')
display(skill_premium)

## 7. Consolidation — the bridge layer
This is where four unrelated files become one story, using **shared constructs** instead of keys:
- **Seniority:** JDS = junior supply, SDS = senior supply, both job files carry seniority via title or experience.
- **Skills:** the five JDS dimensions are re-found in the Analytics postings.
- **Reward:** salary hike (JDS) is set against salary levels and premiums in the job files.
- **Personality (proxy):** SDS traits are set against soft-skill language in senior postings (exploratory).

In [ ]:
# ---- 7a. Market by seniority, side by side ----
mk_dsj = dsj_core.groupby('seniority').agg(dsj_rows=('job_title', 'size'), dsj_median_salary=('avg_salary_lpa', 'median'),
                                          dsj_median_min_exp=('min_experience', 'median'), dsj_openings=('num_of_jobs', 'sum'))
mk_an = an_core.groupby('seniority').agg(an_postings=('s_no', 'size'), an_median_salary_mid=('salary_mid', 'median'),
                                        an_median_min_exp=('exp_min_yrs', 'median'))
market_by_seniority = pd.concat([mk_dsj, mk_an], axis=1)
display(market_by_seniority)

# ---- 7b. Skill demand by seniority (all postings and core data roles) ----
skill_cols = [f'skill_{k}' for k in SKILL_PATTERNS]
dem_all  = an_core.groupby('seniority')[skill_cols].mean().T.rename(index=lambda s: s.replace('skill_', ''))
dem_core = an_core[an_core.is_core_data_role == 1].groupby('seniority')[skill_cols].mean().T.rename(index=lambda s: s.replace('skill_', ''))
skill_demand_by_seniority = pd.concat({'all_postings': dem_all, 'core_data_roles': dem_core}, axis=1)
print('Share of postings mentioning each skill dimension:'); display(skill_demand_by_seniority)

# ---- 7c. Skill alignment table: what predicts junior success vs what the market asks for and pays ----
alignment = pd.DataFrame({
    'jds_cohens_d':      jds_effects.cohens_d,
    'jds_auc_single':    jds_effects.auc_single,
    'demand_all':        an_core[skill_cols].mean().rename(lambda s: s.replace('skill_', '')),
    'demand_junior':     an_core[an_core.seniority == 'Junior'][skill_cols].mean().rename(lambda s: s.replace('skill_', '')),
    'salary_premium_junior_LPA': skill_premium.diff_Junior,
}).loc[JDS_SKILLS]
alignment['effect_rank'] = alignment.jds_cohens_d.rank(ascending=False)
alignment['demand_rank'] = alignment.demand_junior.rank(ascending=False)
alignment['rank_gap'] = alignment.demand_rank - alignment.effect_rank   # >0 : matters for outcomes more than postings suggest
display(alignment)

plt.figure(figsize=(7.5, 5.5))
sns.scatterplot(data=alignment.reset_index(), x='demand_junior', y='jds_cohens_d', s=160)
for k, r in alignment.iterrows():
    plt.annotate(pretty(k), (r.demand_junior, r.jds_cohens_d), xytext=(6, 6), textcoords='offset points')
plt.xlabel('Share of junior postings mentioning the skill (proxy)'); plt.ylabel("JDS effect size on salary hike (Cohen's d)")
plt.title('Skill importance inside the company vs skill demand in the market')
savefig('skill_alignment_scatter')
print('Caution: demand shares are keyword proxies on truncated text; treat the gap as a hypothesis to test, not a conclusion.')

In [ ]:
# ---- 7d. Personality proxy bridge (exploratory) ----
SOFT_PATTERNS = {
    'extraversion':           r'communicat|stakeholder|client|customer|presentation|interpersonal',
    'conscientiousness':      r'attention to detail|detail[- ]oriented|accura|deadline|ownership|organi[sz]ed|quality',
    'openness_to_experience': r'innovat|creativ|curious|research|learn|explor',
    'agreeableness':          r'team player|collaborat|cooperat|teamwork|supportive',
    'neuroticism':            r'pressure|stress|fast[- ]paced|resilien',
}
desc = an_core[an_core.job_description.notna()]
dtext = desc.job_description.str.lower()
soft = pd.DataFrame({k: dtext.str.contains(p, regex=True).astype(int) for k, p in SOFT_PATTERNS.items()}, index=desc.index)
soft['seniority'] = desc.seniority
soft_by_sen = soft.groupby('seniority').mean().T.reindex(SDS_TRAITS)
bridge_personality = pd.concat([sds_effects.cohens_d.reindex(SDS_TRAITS).rename('sds_cohens_d'), soft_by_sen], axis=1)
print(f'Soft-skill language in postings with a description (n={len(desc):,}); proxy mapping is an assumption:')
display(bridge_personality)

## 8. Multivariate structure — PCA and clustering of profiles

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

def pca_block(df, cols, target_col, tag):
    Z = StandardScaler().fit_transform(df[cols])
    pca = PCA(random_state=RANDOM_STATE).fit(Z)
    sc = pca.transform(Z)
    evr = pd.Series(pca.explained_variance_ratio_, index=[f'PC{i + 1}' for i in range(len(cols))])
    load = pd.DataFrame(pca.components_.T, index=[pretty(c) for c in cols], columns=evr.index)
    print(f'{tag}: explained variance'); display(evr.to_frame('share'))
    print(f'{tag}: loadings (PC1-PC3)'); display(load.iloc[:, :3])
    fig, ax = plt.subplots(1, 2, figsize=(12, 4.5))
    ax[0].bar(evr.index, evr.values); ax[0].set_title(f'{tag}: variance explained')
    sns.scatterplot(x=sc[:, 0], y=sc[:, 1], hue=df[target_col].values, ax=ax[1], palette='Set1')
    ax[1].set_xlabel('PC1'); ax[1].set_ylabel('PC2'); ax[1].set_title(f'{tag}: PC1 vs PC2 by outcome')
    savefig(f'{tag.lower()}_pca')

def cluster_block(df, cols, outcome_col, tag, k_range=range(2, 7)):
    Z = StandardScaler().fit_transform(df[cols])
    sil = {k: silhouette_score(Z, KMeans(n_clusters=k, n_init=10, random_state=RANDOM_STATE).fit_predict(Z)) for k in k_range}
    best_k = max(sil, key=sil.get)
    print(f'{tag}: silhouette by k ->', {k: round(v, 3) for k, v in sil.items()}, f'| chosen k={best_k}')
    labels = KMeans(n_clusters=best_k, n_init=10, random_state=RANDOM_STATE).fit_predict(Z)
    out = df.copy(); out['cluster'] = labels
    prof = out.groupby('cluster')[cols].mean(); prof['n'] = out.groupby('cluster').size(); prof['pct_high'] = out.groupby('cluster')[outcome_col].mean() * 100
    print(f'{tag}: cluster profiles (pct_high = % in the high-outcome group)'); display(prof)
    plt.figure(figsize=(8, 3 + 0.4 * best_k))
    zprof = (out.groupby('cluster')[cols].mean() - df[cols].mean()) / df[cols].std()
    sns.heatmap(zprof.rename(columns=pretty), annot=True, fmt='.2f', cmap='coolwarm', center=0); plt.title(f'{tag}: cluster profile (z-scores)')
    savefig(f'{tag.lower()}_clusters')
    return labels

pca_block(jds_core, JDS_SKILLS, 'outcome', 'JDS')
jds_core['cluster'] = cluster_block(jds_core, JDS_SKILLS, 'salary_hike_high_or_low', 'JDS')
pca_block(sds_core, SDS_TRAITS, 'outcome', 'SDS')
sds_core['cluster'] = cluster_block(sds_core, SDS_TRAITS, 'success_classification_high_low', 'SDS')

## 9. Predictive baselines
Small samples (139 and 161 rows) mean **repeated stratified cross-validation**, simple models, and a majority-class baseline. Sensitivity checks re-run the logistic model without flagged rows.

In [ ]:
from sklearn.model_selection import RepeatedStratifiedKFold, KFold, cross_validate, train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.linear_model import LogisticRegression, Ridge
from sklearn.dummy import DummyClassifier, DummyRegressor
from sklearn.tree import DecisionTreeClassifier, export_text
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingRegressor
from sklearn.inspection import permutation_importance
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder

def eval_classifier(X, y, label):
    cv = RepeatedStratifiedKFold(n_splits=5, n_repeats=10, random_state=RANDOM_STATE)
    models = {
        'Baseline (majority)':     DummyClassifier(strategy='most_frequent'),
        'Logistic (standardised)': make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000)),
        'Decision tree (depth 3)': DecisionTreeClassifier(max_depth=3, random_state=RANDOM_STATE),
        'Random forest':           RandomForestClassifier(n_estimators=300, min_samples_leaf=3, random_state=RANDOM_STATE, n_jobs=-1),
    }
    rows = []
    for n, m in models.items():
        r = cross_validate(m, X, y, cv=cv, scoring={'auc': 'roc_auc', 'acc': 'accuracy'}, n_jobs=-1)
        rows.append({'model': n, 'cv_auc': r['test_auc'].mean(), 'auc_sd': r['test_auc'].std(), 'cv_accuracy': r['test_acc'].mean()})
    res = pd.DataFrame(rows).set_index('model')
    print(f'{label}: repeated 5-fold CV (10 repeats)'); display(res)
    return res

def explain_classifier(X, y, label):
    lr = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000)).fit(X, y)
    coef = pd.DataFrame({'std_coef': lr[-1].coef_[0], 'odds_ratio_per_1sd': np.exp(lr[-1].coef_[0])}, index=X.columns).sort_values('std_coef', ascending=False)
    print(f'{label}: logistic coefficients (per 1 SD)'); display(coef)
    Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.3, stratify=y, random_state=RANDOM_STATE)
    rf = RandomForestClassifier(n_estimators=300, min_samples_leaf=3, random_state=RANDOM_STATE, n_jobs=-1).fit(Xtr, ytr)
    pi = permutation_importance(rf, Xte, yte, n_repeats=50, random_state=RANDOM_STATE, scoring='roc_auc')
    imp = pd.DataFrame({'perm_importance': pi.importances_mean, 'sd': pi.importances_std}, index=X.columns).sort_values('perm_importance', ascending=False)
    print(f'{label}: random-forest permutation importance (30% hold-out)'); display(imp)
    tree = DecisionTreeClassifier(max_depth=3, random_state=RANDOM_STATE).fit(X, y)
    print(f'{label}: depth-3 tree rules'); print(export_text(tree, feature_names=list(X.columns)))
    return coef, imp

def quick_auc(X, y):
    cv = RepeatedStratifiedKFold(n_splits=5, n_repeats=10, random_state=RANDOM_STATE)
    return cross_validate(make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000)), X, y, cv=cv, scoring='roc_auc')['test_score'].mean()

# ---- JDS ----
Xj, yj = jds_core[JDS_SKILLS], jds_core.salary_hike_high_or_low
jds_cv = eval_classifier(Xj, yj, 'JDS: predict high salary hike')
jds_coef, jds_imp = explain_classifier(Xj, yj, 'JDS')
clean_j = jds_core[['flag_id_collision', 'flag_repeat_score_profile', 'flag_outlier_z3']].sum(axis=1) == 0
print(f'JDS sensitivity: logistic AUC all rows={quick_auc(Xj, yj):.3f} | excluding flagged rows (n={clean_j.sum()})={quick_auc(Xj[clean_j], yj[clean_j]):.3f}')

In [ ]:
# ---- SDS ----
Xs, ys = sds_core[SDS_TRAITS], sds_core.success_classification_high_low
sds_cv = eval_classifier(Xs, ys, 'SDS: predict high success')
sds_coef, sds_imp = explain_classifier(Xs, ys, 'SDS')
clean_s = sds_core[['flag_id_collision', 'flag_repeat_score_profile', 'flag_outlier_z3']].sum(axis=1) == 0
print(f'SDS sensitivity: logistic AUC all rows={quick_auc(Xs, ys):.3f} | excluding flagged rows (n={clean_s.sum()})={quick_auc(Xs[clean_s], ys[clean_s]):.3f}')

In [ ]:
# ---- Salary models for the two job files (regression baselines) ----
def eval_regressor(df, num_cols, cat_cols, target, label, log_target=False):
    X = df[num_cols + cat_cols]
    y = np.log(df[target]) if log_target else df[target]
    pre = ColumnTransformer([('num', 'passthrough', num_cols),
                             ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), cat_cols)])
    models = {
        'Baseline (mean)':   DummyRegressor(),
        'Ridge':             make_pipeline(pre, StandardScaler(), Ridge(alpha=1.0)),
        'Gradient boosting': make_pipeline(pre, HistGradientBoostingRegressor(random_state=RANDOM_STATE)),
    }
    cv = KFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
    rows = []
    for n, m in models.items():
        r = cross_validate(m, X, y, cv=cv, scoring={'r2': 'r2', 'mae': 'neg_mean_absolute_error'})
        rows.append({'model': n, 'cv_r2': r['test_r2'].mean(), 'cv_mae': -r['test_mae'].mean()})
    print(f'{label} (target={"log " if log_target else ""}{target})'); display(pd.DataFrame(rows).set_index('model'))
    Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.3, random_state=RANDOM_STATE)
    gb = models['Gradient boosting'].fit(Xtr, ytr)
    pi = permutation_importance(gb, Xte, yte, n_repeats=10, random_state=RANDOM_STATE, scoring='r2')
    display(pd.Series(pi.importances_mean, index=X.columns).sort_values(ascending=False).to_frame('permutation_importance (R2 drop)'))

eval_regressor(dsj_core, ['min_experience', 'is_senior_title', 'log_jobs', 'large_recruiter'], ['role_family'],
               'avg_salary_lpa', 'DataScience Jobs salary model', log_target=True)

an_num_feats = ['exp_min_yrs', 'is_metro', 'n_skill_dims'] + skill_cols
eval_regressor(an_core, an_num_feats, ['role_family', 'location_group'], 'salary_mid', 'Analytics Jobs salary model')
print('Read these as descriptive drivers of advertised salary, not causal effects.')

## 10. Export — analysis-ready tables, figures, and a zip

In [ ]:
exports = {
    'jds_core': jds_core, 'sds_core': sds_core, 'dsj_core': dsj_core,
    'analytics_core_no_text': an_core.drop(columns=['job_description']),   # smaller file; text not needed downstream
    'bridge_market_by_seniority': market_by_seniority.reset_index(),
    'bridge_skill_demand_by_seniority': skill_demand_by_seniority.reset_index(),
    'bridge_skill_alignment': alignment.reset_index(),
    'bridge_personality_proxy': bridge_personality.reset_index(),
    'table_jds_effects': jds_effects.reset_index(),
    'table_sds_effects': sds_effects.reset_index(),
    'table_skill_salary_premium': skill_premium.reset_index(),
    'table_dsj_senior_premium': dsj_premium.reset_index(),
}
for name, df in exports.items():
    df.to_csv(os.path.join(OUT_DIR, f'{name}.csv'), index=False)

shutil.make_archive('dd_foundation_outputs', 'zip', OUT_DIR)
print('Saved', len(exports), 'tables +', len(os.listdir(FIG_DIR)), 'figures to', OUT_DIR, '-> dd_foundation_outputs.zip')
if IN_COLAB:
    colab_files.download('dd_foundation_outputs.zip')

In [ ]:
# ---- one-screen recap, computed from the objects above ----
top_skill = jds_effects.cohens_d.idxmax(); top_trait = sds_effects.cohens_d.idxmax(); flat_trait = sds_effects.cohens_d.abs().idxmin()
print('=' * 78); print('FOUNDATION RECAP'); print('=' * 78)
print(f'Core rows -> JDS {len(jds_core)}, SDS {len(sds_core)}, DS Jobs {len(dsj_core)}, Analytics {len(an_core):,}')
print(f'Strongest junior skill signal : {pretty(top_skill)} (d={jds_effects.cohens_d.max():.2f})')
print(f'Strongest senior trait signal : {pretty(top_trait)} (d={sds_effects.cohens_d.max():.2f}); weakest: {pretty(flat_trait)} (d={sds_effects.cohens_d[flat_trait]:.2f})')
print(f'JDS model CV AUC (logistic) {jds_cv.loc["Logistic (standardised)", "cv_auc"]:.2f} | SDS {sds_cv.loc["Logistic (standardised)", "cv_auc"]:.2f}')
if 'senior_premium_pct' in dsj_premium:
    print(f'Median senior salary premium across role families: {dsj_premium.senior_premium_pct.median():.0f}%')
gap = alignment.rank_gap.idxmax()
print(f'Largest importance-vs-demand gap: {pretty(gap)} (rank gap {alignment.rank_gap.max():+.0f})')

## Where to go from here
1. Pick the objective (for example *talent-market alignment by career stage*) and keep sections 7–9 as the core of the analysis chapter.
2. Tighten the skill dictionaries in section 4 (check a sample of postings by hand) and report keyword coverage.
3. Add objective-specific modelling on top of the exported `bridge_*` tables.
4. State the limitations explicitly: no row-level linkage, single-company trait samples, keyword proxies on truncated text, associational results only.